<h3 style="color:#6FA8DC; font-weight:bold">Handling Dates and Time</h3>

Dates and time are very common in real-world ML datasets.

A date like `2019-12-10` looks simple, but an ML model usually cannot directly understand it as useful information. We often **extract meaningful features** from it such as year, month, day, day of week, weekend, quarter, and time-related values.

This notebook follows the reference notebook `working-with-dates-and-time.ipynb` and uses the provided datasets `orders.csv` and `messages.csv` directly from the same folder.

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# Dataset files are in the same folder as this notebook
orders = pd.read_csv('orders.csv')
messages = pd.read_csv('messages.csv')

orders.head()

In [ ]:
messages.head()

<h5 style="color:#78B89A; font-weight:bold;">Understanding the datasets →</h5>

- `orders.csv` contains an `date` column with dates and an `orders` column.
- `messages.csv` contains a `date` column with both date and time.

The first important step is to convert the date column into Pandas **datetime datatype**.

In [ ]:
orders.info()
messages.info()

<h3 style="color:#6FA8DC; font-weight:bold">1. Converting to Datetime</h3>

CSV files usually load dates as `object`/string values. We use `pd.to_datetime()` to convert them into a proper datetime datatype.

⭐ **VVVV Important:** After conversion, we can use the `.dt` accessor to extract date/time features.

In [ ]:
orders['date'] = pd.to_datetime(orders['date'])
messages['date'] = pd.to_datetime(messages['date'])

orders.info()
messages.info()

<h3 style="color:#6FA8DC; font-weight:bold">2. Extract Year</h3>

`dt.year` extracts the year from a date.

Example: `2019-12-10 → 2019`

This can be useful when the behavior of a business changes over different years.

In [ ]:
orders['date_year'] = orders['date'].dt.year
orders[['date', 'date_year']].sample(5)

<h3 style="color:#6FA8DC; font-weight:bold">3. Extract Month</h3>

We can extract the month as:

- month number → `dt.month`
- month name → `dt.month_name()`

Example: `2019-12-10 → 12 → December`

In [ ]:
orders['date_month_no'] = orders['date'].dt.month
orders[['date', 'date_month_no']].head()

In [ ]:
orders['date_month_name'] = orders['date'].dt.month_name()
orders[['date', 'date_month_no', 'date_month_name']].head()

<h3 style="color:#6FA8DC; font-weight:bold">4. Extract Day</h3>

`dt.day` gives the day of the month.

Example: `2019-12-10 → 10`

In [ ]:
orders['date_day'] = orders['date'].dt.day
orders[['date', 'date_day']].head()

<h3 style="color:#6FA8DC; font-weight:bold">5. Day of Week</h3>

`dt.dayofweek` gives a number from `0` to `6`:

- Monday → 0
- Tuesday → 1
- ...
- Sunday → 6

`dt.day_name()` gives the actual day name.

In [ ]:
orders['date_dow'] = orders['date'].dt.dayofweek
orders[['date', 'date_dow']].head()

In [ ]:
orders['date_dow_name'] = orders['date'].dt.day_name()
orders[['date', 'date_dow', 'date_dow_name']].head()

<h3 style="color:#6FA8DC; font-weight:bold">6. Is the Date a Weekend?</h3>

Weekend information can be useful in ML because customer behavior, sales, orders, website traffic, etc. can differ between weekdays and weekends.

We create:

- `1` → weekend
- `0` → not weekend

In [ ]:
orders['date_is_weekend'] = np.where(
    orders['date_dow_name'].isin(['Saturday', 'Sunday']),
    1,
    0
)

orders[['date', 'date_dow_name', 'date_is_weekend']].head()

<h3 style="color:#6FA8DC; font-weight:bold">7. Extract Week of the Year</h3>

A year contains approximately 52 weeks. Sometimes the week number is more useful than the exact date.

⭐ Modern Pandas approach: use `dt.isocalendar().week`.

The older reference notebook uses `dt.week`, but that approach is deprecated in modern Pandas.

In [ ]:
orders['date_week'] = orders['date'].dt.isocalendar().week.astype('int64')
orders[['date', 'date_week']].head()

<h3 style="color:#6FA8DC; font-weight:bold">8. Extract Quarter</h3>

A year is divided into 4 quarters:

- Q1 → January to March
- Q2 → April to June
- Q3 → July to September
- Q4 → October to December

`dt.quarter` gives the quarter number.

In [ ]:
orders['quarter'] = orders['date'].dt.quarter
orders[['date', 'quarter']].head()

<h3 style="color:#6FA8DC; font-weight:bold">9. Extract Semester / Half-Year</h3>

We can divide the year into two halves:

- Semester 1 → Q1 + Q2
- Semester 2 → Q3 + Q4

In [ ]:
orders['semester'] = np.where(orders['quarter'].isin([1, 2]), 1, 2)
orders[['date', 'quarter', 'semester']].head()

<h3 style="color:#6FA8DC; font-weight:bold">10. Complete Date Features</h3>

Now one date has been converted into many useful ML features.

```text
Date
 ↓
Year
Month
Month Name
Day
Day of Week
Day Name
Weekend
Week of Year
Quarter
Semester
```

This is the main idea of **date feature engineering**.

In [ ]:
date_features = [
    'date', 'date_year', 'date_month_no', 'date_month_name',
    'date_day', 'date_dow', 'date_dow_name', 'date_is_weekend',
    'date_week', 'quarter', 'semester'
]

orders[date_features].head()

<h3 style="color:#6FA8DC; font-weight:bold">11. Time Elapsed Between Dates</h3>

We can calculate how much time has passed between a date in our dataset and another date.

For example:

```text
Today - Order Date = How old is this order?
```

The reference notebook uses the current date/time. Here we use the same idea.

In [ ]:
import datetime

today = datetime.datetime.today()
today

In [ ]:
today - orders['date']

In [ ]:
(today - orders['date']).dt.days

<h5 style="color:#78B89A; font-weight:bold;">Elapsed days →</h5>

`dt.days` converts the timedelta into the number of complete days.

In [ ]:
orders['days_since_order'] = (today - orders['date']).dt.days
orders[['date', 'days_since_order']].head()

<h5 style="color:#78B89A; font-weight:bold;">Elapsed months →</h5>

For approximate month differences, the reference notebook uses NumPy's month timedelta.

In [ ]:
months_passed = np.round(
    (today - orders['date']) / np.timedelta64(1, 'M'),
    0
)

months_passed.head()

<h3 style="color:#6FA8DC; font-weight:bold">12. Working with Time</h3>

The `messages.csv` dataset contains values such as:

```text
2013-12-15 00:50:00
```

This contains both:

- Date → `2013-12-15`
- Time → `00:50:00`

After converting the column to datetime, we can extract the time components.

In [ ]:
messages.info()

messages['date'] = pd.to_datetime(messages['date'])
messages.info()

<h3 style="color:#6FA8DC; font-weight:bold">13. Extract Hour, Minute and Second</h3>

⭐ **VVVV Important:** These are useful when behavior depends on the time of day.

For example, messages/orders may behave differently during morning, afternoon, evening, and night.

In [ ]:
messages['hour'] = messages['date'].dt.hour
messages['min'] = messages['date'].dt.minute
messages['sec'] = messages['date'].dt.second

messages[['date', 'hour', 'min', 'sec']].head()

<h3 style="color:#6FA8DC; font-weight:bold">14. Extract the Time Part</h3>

`dt.time` extracts only the time from a datetime value.

In [ ]:
messages['time'] = messages['date'].dt.time
messages[['date', 'time']].head()

<h3 style="color:#6FA8DC; font-weight:bold">15. Time Difference</h3>

Just like dates, we can calculate how much time has passed since a timestamp.

In [ ]:
today - messages['date']

<h5 style="color:#78B89A; font-weight:bold;">Convert the difference into seconds →</h5>

In [ ]:
(today - messages['date']) / np.timedelta64(1, 's')

<h5 style="color:#78B89A; font-weight:bold;">Convert the difference into minutes →</h5>

In [ ]:
(today - messages['date']) / np.timedelta64(1, 'm')

<h5 style="color:#78B89A; font-weight:bold;">Convert the difference into hours →</h5>

In [ ]:
(today - messages['date']) / np.timedelta64(1, 'h')

<h3 style="color:#6FA8DC; font-weight:bold">16. Practical ML Feature Engineering Example</h3>

Suppose we want to predict the number of orders. Instead of giving the raw date directly to the model, we can create useful features from it.

In [ ]:
ml_features = orders[[
    'date_year',
    'date_month_no',
    'date_day',
    'date_dow',
    'date_is_weekend',
    'date_week',
    'quarter',
    'semester'
]].copy()

ml_features['orders'] = orders['orders']
ml_features.head()

<h3 style="color:#6FA8DC; font-weight:bold">17. Why Do We Handle Dates and Time?</h3>

A raw date is often not the most useful form for an ML model.

For example:

```text
2019-12-10
```

can become:

```text
Year = 2019
Month = 12
Day = 10
Day of Week = Tuesday
Weekend = 0
Week = 50
Quarter = 4
Semester = 2
```

These extracted values can capture **seasonality, trends, weekly patterns and time-based behavior**.

<h3 style="color:#6FA8DC; font-weight:bold">18. Important `.dt` Functions to Remember</h3>

| Function | Meaning |
|---|---|
| `dt.year` | Year |
| `dt.month` | Month number |
| `dt.month_name()` | Month name |
| `dt.day` | Day of month |
| `dt.dayofweek` | Day number (Monday = 0) |
| `dt.day_name()` | Day name |
| `dt.isocalendar().week` | Week number |
| `dt.quarter` | Quarter |
| `dt.hour` | Hour |
| `dt.minute` | Minute |
| `dt.second` | Second |
| `dt.time` | Time part |

⭐ **VVVV Important:** First convert the column using `pd.to_datetime()`, then use `.dt`.

<h3 style="color:#6FA8DC; font-weight:bold">19. Final Revision Flow</h3>

```text
Raw CSV Date / Timestamp
          ↓
   pd.to_datetime()
          ↓
       .dt
          ↓
 ┌────────┼─────────────┐
 ↓        ↓             ↓
Year    Month       Day/Week
 ↓        ↓             ↓
Quarter  Weekend     Time
          ↓
   Useful ML Features
          ↓
        Model
```

### Remember

- `pd.to_datetime()` → converts string/date column to datetime
- `.dt` → accesses datetime properties
- `dt.year` → year
- `dt.month` → month
- `dt.day` → day
- `dt.dayofweek` → weekday number
- `dt.day_name()` → weekday name
- `dt.isocalendar().week` → week number
- `dt.quarter` → quarter
- `dt.hour`, `dt.minute`, `dt.second` → time components
- `dt.time` → time only
- subtraction of datetime values → `timedelta`

**Main idea:** Convert dates into meaningful numerical/categorical features that an ML model can use.